# Run D0

Quasi-stationary validation set D_0 (ID 1281). Window 2026-07-30 14:43:30 to 2026-07-30 23:24:25, output interval 1 s.

Model: VFCS `5fd49f1` with `subSystemAir = SubSystemAirCMP` and `subSystemCooling = SubSystemCoolingTST` in `FuelCellSystem`. Solver DASSL, tolerance 1e-4.

Workflow: measured data, simulation input, stored simulation result, comparison on the measured time base, evaluation after warm-up.

## 1. Setup and imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(root))

from src.io import data_processing as dp
from src.io import data_visualization as dv
from src.io import run_workflow as rw
from src.io import validation_figures as vf
from src.io.runtime_paths import resolve_results_dir

pd.set_option('display.max_columns', 30, 'display.width', 160)

## 2. Run configuration

`WRITE_INPUT` stays `False` unless this run is about to be simulated. With `True`, section 4 overwrites the single input table read by the server.

In [ ]:
RUN_ID = 'D0'
STEP_SIZE = 1
WRITE_INPUT = False

config = rw.load_run_config(RUN_ID)
source = config.data_source
skip_time = source.skip_time
pd.Series({
    'Start': source.start_date,
    'End': source.end_date,
    'Time axis': source.time_axis,
    'Warm-up skipped [s]': skip_time,
    'Inputs': len(source.input_variables),
    'Outputs': len(source.output_variables),
}).to_frame('Value')

## 3. Measured data

Inputs and outputs of the window on the elapsed time axis.

In [ ]:
input_df, output_df = rw.load_measurements(config)
print(f'Inputs {input_df.shape}, outputs {output_df.shape}, end of window {input_df.index.max():.1f} s')
print('Sample interval [s]:')
display(pd.Series(input_df.index).diff().describe().round(3).to_frame('dt').T)
display(input_df.describe().T.round(3))
display(output_df.describe().T.round(3))

In [ ]:
fig = vf.plot_signals(input_df)

## 4. Simulation input

The table as it enters the simulation. It is only written with `WRITE_INPUT = True`.

In [ ]:
table_df = rw.simulation_table(input_df)
stop_time = float(table_df.index.max())
display(table_df.head())
print(f'Simulation setup: stop time {stop_time} s, interval 1 s, tolerance 1e-4, DASSL')

if WRITE_INPUT:
    dp.prepare_sim_input(config, table_df)
    print('Input table written.')
else:
    print('Input table not written.')

## 5. Stored simulation result

Simulate the model and store the result first. This section loads the latest stored attempt.

In [ ]:
result_path = rw.latest_result(RUN_ID)
display(rw.stored_run_summary(result_path).to_frame('Value'))

sim_df = rw.load_simulation(config, result_path, STEP_SIZE)
print(f'Simulated signals {sim_df.shape}, end {sim_df.index.max():.1f} s')

## 6. Comparison on the measured time base

System quantities are added and the simulation is sampled at the measured times. The shaded span is the skipped warm-up.

In [ ]:
sim_df, source_df = rw.align_comparison(config, sim_df, output_df)
fig = vf.plot_time_comparison(sim_df, source_df, skip_time)

In [ ]:
sim_eval, source_eval = sim_df.loc[skip_time:], source_df.loc[skip_time:]
print(f'Samples after warm-up: {len(source_eval)} of {len(source_df)}')

## 7. Grouping

The last valid step of each dwell is kept. Each group is the mean over the true window T_avg.

In [ ]:
results = rw.compare_stationary(config, sim_eval, source_eval)
trigger = config.group_config.trigger
print(f"Groups: {len(results['measurement'])}, T_avg {config.group_config.group_time} s")
print(f"Samples per group: {results['n_samples'].min()} to {results['n_samples'].max()}")
display(results['measurement'].tail().round(3))
display(results['simulation'].tail().round(3))

In [ ]:
dv.plot_mult_over_one(
    [results['simulation'], results['measurement']],
    x_col=trigger,
    legend=['Simulation', 'Experiment'],
    fontsize=12,
    data_size=30,
)
plt.show()

## 8. Type A uncertainty of the group means

u_A = s / sqrt(n) with the sample standard deviation s of each group.

In [ ]:
columns = [c for c in results['std'].columns if c != trigger]
pd.DataFrame({
    's median': results['std'][columns].median(),
    's max': results['std'][columns].max(),
    'u_A median': results['u_A'][columns].median(),
    'u_A max': results['u_A'][columns].max(),
}).round(4)

## 9. Passes per operating condition

Each condition was run twice. The table shows the difference between the two passes. The metrics below use the mean over both passes.

In [ ]:
print(f"Operating conditions: {len(results['measurement_conditions'])}")
results['pass_difference'].abs().describe().T[['mean', 'max']].round(4)

## 10. Error metrics

In [ ]:
results['metrics'].round(4)

## 11. Simulation over experiment

The dashed line marks agreement.

In [ ]:
fig = vf.plot_stationary_comparison(results['simulation_conditions'], results['measurement_conditions'])

## 12. Uncertainty of the accuracy assessment

u_D combines the Type B uncertainty of the measurement chain with the Type A uncertainty of the means. A residual is significant when |e| > k u_D with k = 2.

In [ ]:
from src.io.measurement_uncertainty import evaluate_uncertainty, load_budget, relative_residuals
from src.io.runtime_paths import resolve_config_path

budget = load_budget(resolve_config_path('measurement_uncertainty.yml'))
conditions = pd.read_csv(resolve_config_path(config.group_config.conditions_file))
uncertainty_table, uncertainty_summary = evaluate_uncertainty(results, conditions, trigger, budget)
display(uncertainty_summary)
display(uncertainty_table.head())

## 13. Details of the Result Evaluation

Largest sample standard deviation of the windows, pass differences of experiment and simulation, errors per pass, and range and load dependence of the relative residual.

In [ ]:
details = rw.stationary_details(results, conditions, trigger)
details['relative_residuals'] = relative_residuals(uncertainty_table)
for name, frame in details.items():
    print(name)
    display(frame.round(4))

## 14. Export tables and figure

In [ ]:
results_dir = resolve_results_dir()
figure_dir = results_dir / 'figures'
figure_dir.mkdir(exist_ok=True)

results['metrics'].to_csv(results_dir / 'S01_metrics.csv')
uncertainty_table.to_csv(results_dir / 'S01_uncertainty_conditions.csv', index=False)
uncertainty_summary.to_csv(results_dir / 'S01_uncertainty_summary.csv')
for name, frame in details.items():
    frame.to_csv(results_dir / f'S01_{name}.csv')
fig.savefig(figure_dir / 'fcs_d0_group.png', dpi=300, bbox_inches='tight')